# Etapas 2 a 4 e 7 - Preparação, baseline, Thompson Sampling e MLflow

Este notebook prepara a base, cria uma política determinística de controle, executa Thompson Sampling, compara métricas, gera o Golden Set com cinco casos e registra o experimento no MLflow.

A base pública não registra qual oferta foi apresentada em cada contato. Para demonstrar o algoritmo, este projeto cria três braços proxy usando tercis de idade calculados no conjunto de treino. Isso não representa um experimento causal nem uma política contextual: as features dos clientes não condicionam as recomendações atuais.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'src' / 'utils.py').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATA_FILE, MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, RANDOM_SEED,
    TEST_SIZE, THOMPSON_ALPHA_PRIOR, THOMPSON_BETA_PRIOR,
    THOMPSON_NUM_ARMS, RANDOM_STATE
)
from src.models import BaselinePolicy, ThompsonSampling, evaluate_strategy
from src.utils import (
    clean_and_prepare_data, create_golden_set, encode_features,
    load_data, scale_features, split_data
)

np.random.seed(RANDOM_SEED)
print(f'Raiz do projeto: {PROJECT_ROOT}')
print(f'Arquivo de dados: {DATA_FILE}')

Raiz do projeto: c:\Users\cvmar\OneDrive\Área de Trabalho\datathon-8mlet-grupo-35
Arquivo de dados: c:\Users\cvmar\OneDrive\Área de Trabalho\datathon-8mlet-grupo-35\data\bank-marketing.csv


In [2]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f'Base nao encontrada em {DATA_FILE}. Execute o download indicado no README.'
    )

df = load_data(str(DATA_FILE))
df_clean = clean_and_prepare_data(df, drop_duration=True)
X_encoded, y_encoded, feature_names, encoders = encode_features(df_clean, target_col='y')
X_train, X_test, y_train, y_test = split_data(
    X_encoded, y_encoded, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

print(f'Features: {len(feature_names)}')
print(f'Treino: {X_train_scaled.shape[0]:,} linhas')
print(f'Teste: {X_test_scaled.shape[0]:,} linhas')
print(f'Taxa de conversao no treino: {y_train.mean():.2%}')
print(f'Taxa de conversao no teste: {y_test.mean():.2%}')
display(pd.DataFrame({'feature': feature_names}))

Features: 19
Treino: 31,523 linhas
Teste: 7,881 linhas
Taxa de conversao no treino: 11.67%
Taxa de conversao no teste: 11.67%


,feature
0,age
1,job
2,marital
3,education
4,default
5,housing
6,loan
7,contact
8,month
9,day_of_week


## Criação dos braços proxy

Como a base pública não informa qual oferta foi apresentada em cada contato, usamos três braços proxy, definidos pelos tercis de idade no conjunto de treino. Isso demonstra a mecânica do bandit, mas não equivale a um experimento causal com ofertas reais.

In [3]:
age_index = feature_names.index('age')
arm_thresholds = np.quantile(X_train[:, age_index], [1 / 3, 2 / 3])
y_train_arms = np.digitize(X_train[:, age_index], arm_thresholds)
y_test_arms = np.digitize(X_test[:, age_index], arm_thresholds)

print(f'Indice da feature age: {age_index}')
print(f'Limiares dos bracos: {arm_thresholds}')
print('Distribuicao dos bracos no treino:', np.bincount(y_train_arms, minlength=THOMPSON_NUM_ARMS))
print('Distribuicao dos bracos no teste:', np.bincount(y_test_arms, minlength=THOMPSON_NUM_ARMS))

Indice da feature age: 0
Limiares dos bracos: [34. 44.]
Distribuicao dos bracos no treino: [ 9799 10861 10863]
Distribuicao dos bracos no teste: [2533 2719 2629]


## Etapa 3 - Baseline determinístico

In [4]:
baseline = BaselinePolicy(num_arms=THOMPSON_NUM_ARMS)
baseline.fit(y_train, y_train_arms)
baseline_predictions = baseline.predict(X_test_scaled)
baseline_metrics = evaluate_strategy(y_test, baseline_predictions)

print(f'Melhor braco historico: {baseline.best_arm}')
print(f'Taxa historica do melhor braco: {baseline.get_conversion_rate():.2%}')
display(pd.DataFrame([baseline_metrics], index=['baseline']))

Melhor braco historico: 0
Taxa historica do melhor braco: 14.04%


,conversion_rate,total_reward,exploration_rate,n_samples,n_unique_arms
baseline,0.116736,920,0.0,7881,1


## Etapa 3 - Thompson Sampling

Cada braço recebe uma distribuição Beta iniciada com `alpha_prior=1` e `beta_prior=1`. A política amostra uma taxa de sucesso para cada braço e escolhe a maior amostra.

In [5]:
thompson = ThompsonSampling(
    num_arms=THOMPSON_NUM_ARMS,
    alpha_prior=THOMPSON_ALPHA_PRIOR,
    beta_prior=THOMPSON_BETA_PRIOR,
    random_state=RANDOM_SEED,
)
thompson.fit(y_train, y_train_arms)
thompson_predictions = thompson.predict(X_test_scaled)
thompson_metrics = evaluate_strategy(y_test, thompson_predictions)

arm_summary = pd.DataFrame({
    'braco': range(THOMPSON_NUM_ARMS),
    'alpha': thompson.alpha,
    'beta': thompson.beta,
    'probabilidade_esperada': thompson.alpha / (thompson.alpha + thompson.beta),
})
display(arm_summary)
display(pd.DataFrame([thompson_metrics], index=['thompson_sampling']))

,braco,alpha,beta,probabilidade_esperada
0,0,1377.0,8424.0,0.140496
1,1,1012.0,9851.0,0.093160
2,2,1292.0,9573.0,0.118914


,conversion_rate,total_reward,exploration_rate,n_samples,n_unique_arms
thompson_sampling,0.116736,920,0.0,7881,1


## Etapa 4 - Comparação e Golden Set

In [6]:
comparison = pd.DataFrame([baseline_metrics, thompson_metrics], index=['baseline', 'thompson_sampling'])
comparison['conversion_rate_pct'] = comparison['conversion_rate'] * 100
comparison['total_reward'] = comparison['total_reward'].astype(int)
display(comparison[['conversion_rate_pct', 'total_reward', 'exploration_rate', 'n_unique_arms']])

absolute_gain = thompson_metrics['conversion_rate'] - baseline_metrics['conversion_rate']
relative_gain = (absolute_gain / baseline_metrics['conversion_rate'] * 100) if baseline_metrics['conversion_rate'] else np.nan
print(f'Ganho absoluto: {absolute_gain:+.2%}')
print(f'Ganho relativo: {relative_gain:+.2f}%')

,conversion_rate_pct,total_reward,exploration_rate,n_unique_arms
baseline,11.673645,920,0.0,1
thompson_sampling,11.673645,920,0.0,1


Ganho absoluto: +0.00%
Ganho relativo: +0.00%


In [7]:
np.random.seed(RANDOM_SEED)
X_golden, y_golden, golden_indices = create_golden_set(X_test_scaled, y_test, n_samples=5)
golden_df = pd.DataFrame({
    'cliente_id': [f'C{index:04d}' for index in golden_indices],
    'conversao_real': np.where(y_golden == 1, 'sim', 'nao'),
    'oferta_baseline': baseline_predictions[golden_indices],
    'oferta_thompson': thompson_predictions[golden_indices],
})
display(golden_df)
golden_df.to_csv(PROJECT_ROOT / 'golden_set.csv', index=False)
print('Golden Set salvo em golden_set.csv')

,cliente_id,conversao_real,oferta_baseline,oferta_thompson
0,C2614,sim,0,0
1,C3095,sim,0,0
2,C0277,nao,0,0
3,C3331,nao,0,0
4,C3698,nao,0,0


Golden Set salvo em golden_set.csv


## Etapa 7 - Acompanhamento com MLflow

In [8]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name='notebook_thompson_sampling') as run:
    mlflow.log_param('algoritmo_principal', 'thompson_sampling')
    mlflow.log_param('algoritmo_baseline', 'always_best_arm')
    mlflow.log_param('num_ofertas', THOMPSON_NUM_ARMS)
    mlflow.log_param('alpha_prior', THOMPSON_ALPHA_PRIOR)
    mlflow.log_param('beta_prior', THOMPSON_BETA_PRIOR)
    mlflow.log_param('test_size', TEST_SIZE)
    mlflow.log_param('random_seed', RANDOM_SEED)
    mlflow.log_param('n_samples_treino', X_train.shape[0])
    mlflow.log_param('n_samples_teste', X_test.shape[0])
    mlflow.log_param('n_features', X_train.shape[1])
    mlflow.log_param('kaggle_dataset', 'bank-marketing')
    mlflow.log_metric('baseline_conversion_rate', baseline_metrics['conversion_rate'])
    mlflow.log_metric('baseline_total_reward', baseline_metrics['total_reward'])
    mlflow.log_metric('baseline_exploration_rate', baseline_metrics['exploration_rate'])
    mlflow.log_metric('thompson_conversion_rate', thompson_metrics['conversion_rate'])
    mlflow.log_metric('thompson_total_reward', thompson_metrics['total_reward'])
    mlflow.log_metric('thompson_exploration_rate', thompson_metrics['exploration_rate'])
    mlflow.log_metric('melhoria_conversao_absoluta', absolute_gain)
    mlflow.log_metric('melhoria_conversao_relativa_pct', relative_gain)
    mlflow.log_artifact(str(PROJECT_ROOT / 'golden_set.csv'))
    mlflow.sklearn.log_model(
        thompson,
        'thompson_sampling_model',
        registered_model_name='thompson_sampling',
    )
    print(f'Run ID: {run.info.run_id}')

2026/10/01 12:41:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'thompson_sampling' already exists. Creating a new version of this model...


Run ID: ab94aca959ed4d6994923868b5cd8ecf


Created version '4' of model 'thompson_sampling'.


In [9]:
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
joblib.dump(thompson, models_dir / 'thompson_model.pkl')
joblib.dump(scaler, models_dir / 'scaler.pkl')
print(f'Modelos salvos em {models_dir}')

Modelos salvos em c:\Users\cvmar\OneDrive\Área de Trabalho\datathon-8mlet-grupo-35\models


### Interpretação e limitações

Os resultados deste notebook demonstram o pipeline e a mecânica do Thompson Sampling com a base pública. Como não há registro da oferta realmente apresentada para cada cliente, os braços são proxies e a avaliação não deve ser interpretada como efeito causal de uma campanha real. As recomendações atuais também não são condicionadas pelas features do cliente. Em produção, a política deve registrar a oferta efetivamente exibida e seu resultado antes de estimar impacto; mantenha revisão humana para decisões sensíveis.